In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

In [3]:
%LoadCheckpoint /scratch/jieq/pandax/ds_notebooks/nyc-taxi/src/small_bench/checkpoints/post_cell_9.pickle

trying: ['orig_output']
me:  20
trying: ['trip_data']


me:  17
trying: ['factor']
me:  1
trying: ['file_loc']
me:  1
trying: ['benchmark_name']
me:  1
trying: ['pd']
me:  0
trying: ['map_payment_type']
me:  17
trying: ['Path']
me:  0
trying: ['BENCHMARKS_TO_PATHS']
me:  0


Declaring variable pd
Declaring variable Path
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable factor
Declaring variable file_loc
Declaring variable benchmark_name
Declaring variable trip_data
Declaring variable map_payment_type
Declaring variable orig_output


In [4]:
%%RecordEvent
%%time
### cell 10 ###

# compute total_taxes in one vectorized kernel instead of two binary adds
trip_data['total_taxes'] = trip_data[['extra','mta_tax','improvement_surcharge']].sum(axis=1)
# drop the now redundant columns
gpu_trip_data = trip_data.drop(['extra','mta_tax','improvement_surcharge'], axis=1)
# preview
gpu_trip_data.head()

CPU times: user 34.1 ms, sys: 89.9 ms, total: 124 ms
Wall time: 132 ms


,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,tip_amount,tolls_amount,total_amount,duration,trip_pickup_hour,trip_dropoff_hour,trip_day,total_taxes
0,2018-01-01 00:21:05,2018-01-01 00:24:23,1,0.5,41,24,Cash,4.5,0.0,0.0,5.8,3.3,0,0,Monday,1.3
1,2018-01-01 00:44:55,2018-01-01 01:03:05,1,2.7,239,140,Cash,14.0,0.0,0.0,15.3,18.2,0,1,Monday,1.3
2,2018-01-01 00:08:26,2018-01-01 00:14:21,2,0.8,262,141,Credit_card,6.0,1.0,0.0,8.3,5.9,0,0,Monday,1.3
3,2018-01-01 00:20:22,2018-01-01 00:52:51,1,10.2,140,257,Cash,33.5,0.0,0.0,34.8,32.5,0,0,Monday,1.3
4,2018-01-01 00:09:18,2018-01-01 00:27:06,2,2.5,246,239,Credit_card,12.5,2.8,0.0,16.6,17.8,0,0,Monday,1.3


In [5]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/nyc-taxi/src/rewritten/o4_mini_high_small/checkpoints/post_cell_10_try_2.pickle

migration speed (bps): 860630252.4477155
---------------------------
variables to migrate:
factor 24
Path 904
map_payment_type 144
trip_data 1297389725
BENCHMARKS_TO_PATHS 2272
file_loc 88
orig_output 16
pd 72
gpu_trip_data 1087152749
benchmark_name 57
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /scratch/jieq/pandax/ds_notebooks/nyc-taxi/src/rewritten/o4_mini_high_small/checkpoints/post_cell_10_try_2.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables ['Path', 'BENCHMARKS_TO_PATHS']
Active variables ['trip_data']
Intermediate variables ['benchmark_name', 'file_loc', 'factor']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['trip_data']
Active variables []
Intermediate variables []
Future variables ['trip_data']
Modified dataframes
======= Cell 2 =======
Input variables ['trip_data']
Active variables []
Intermediate variables []
Future variables ['trip_data']
Modified dataframes
======= Cell 3 =======
Input variables ['trip_data']
Active variables ['trip_data']
Intermediate variables []
Future variables []
Modified dataframes
  trip_data
    Input columns: set()
    Changed columns: set()
    Created columns: set()
    Deleted columns: {'store_and_fwd_flag', 'VendorID', 'RatecodeID'}
======= Cell 4 =======
Input variables ['trip_data']
Active variables ['trip_data']
Intermediate variables []
Future variables []
Modified dataframes
  trip_data
    Input columns: se

In [7]:

with open("/scratch/jieq/pandax/ds_notebooks/nyc-taxi/src/opt_cell_exec_info_10_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[10], f)


In [8]:
opt_output = Out.get(4)

In [9]:
trip_data_opt = trip_data
%LoadCheckpoint /scratch/jieq/pandax/ds_notebooks/nyc-taxi/src/small_bench/checkpoints/post_cell_10.pickle
assert compare_df(trip_data_opt, trip_data)

import numpy as np
if os.getenv("USE_GPU") == "True":
    import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
if os.getenv("USE_GPU") == "True":
    is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
    is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
else:
    is_orig_output_array = isinstance(orig_output, np.ndarray)
    is_opt_output_array = isinstance(opt_output, np.ndarray)

is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['trip_data']


me:  21
trying: ['benchmark_name']
me:  1
trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['orig_output']
me:  22
trying: ['factor']
me:  1
trying: ['file_loc']
me:  1
trying: ['Path']
me:  0
trying: ['map_payment_type']
me:  17
trying: ['pd']
me:  0


Declaring variable BENCHMARKS_TO_PATHS
Declaring variable Path
Declaring variable pd
Declaring variable benchmark_name
Declaring variable factor
Declaring variable file_loc
Declaring variable map_payment_type
Declaring variable trip_data
Declaring variable orig_output


ValueError: Shape mismatch: (8759874, 19) vs (8759874, 16)